# Feature Engineering

En este notebook se construye el dataset de entrenamiento para el modelo de pronóstico de demanda, integrando la información histórica de ventas con atributos de productos, tiendas e inventario, además de generar variables temporales e históricas.

## Cargar datos

In [1]:
import sys
from pathlib import Path

# Raíz del proyecto
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [27]:
import importlib
import src.features

importlib.reload(src.features)

<module 'src.features' from 'c:\\Users\\beltr\\Documents\\prueba_tostao\\src\\features.py'>

In [28]:
import pandas as pd
import seaborn as sns
from pathlib import Path
from src.preprocessing import load_data
from src.features import * 

In [3]:
DATA_PATH = Path("../data/raw")

data = load_data(DATA_PATH)

df_ventas = data["ventas"]
df_catalogo = data["catalogo"]
df_inventario = data["inventario"]
df_tiendas = data["tiendas"] 
df_ground = data['ground'] 

### 1. Union de los datasets

In [7]:

df = merge_data(
    df_ventas,
    df_catalogo,
    df_tiendas,
    df_inventario
)

In [8]:
df.head()

,fecha,id_tienda,id_producto,unidades_vendidas,nombre,categoria,costo_unitario,precio_venta,costo_almacenamiento_semanal,ciudad,tamaño_m2,stock_actual
0,2024-01-01,STORE_01,PROD_001,5,Tinto,Bebidas,800,2500,10,Bogotá,28,1
1,2024-01-02,STORE_01,PROD_001,4,Tinto,Bebidas,800,2500,10,Bogotá,28,1
2,2024-01-03,STORE_01,PROD_001,3,Tinto,Bebidas,800,2500,10,Bogotá,28,1
3,2024-01-04,STORE_01,PROD_001,4,Tinto,Bebidas,800,2500,10,Bogotá,28,1
4,2024-01-05,STORE_01,PROD_001,10,Tinto,Bebidas,800,2500,10,Bogotá,28,1


In [9]:
print(df.isna().sum())

fecha                           0
id_tienda                       0
id_producto                     0
unidades_vendidas               0
nombre                          0
categoria                       0
costo_unitario                  0
precio_venta                    0
costo_almacenamiento_semanal    0
ciudad                          0
tamaño_m2                       0
stock_actual                    0
dtype: int64


Se consolidó la información de ventas históricas con los atributos de productos, tiendas e inventario.

El resultado es un único dataset que contiene la información necesaria para construir variables predictoras y entrenar el modelo de pronóstico.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14560 entries, 0 to 14559
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   fecha                         14560 non-null  object
 1   id_tienda                     14560 non-null  object
 2   id_producto                   14560 non-null  object
 3   unidades_vendidas             14560 non-null  int64 
 4   nombre                        14560 non-null  object
 5   categoria                     14560 non-null  object
 6   costo_unitario                14560 non-null  int64 
 7   precio_venta                  14560 non-null  int64 
 8   costo_almacenamiento_semanal  14560 non-null  int64 
 9   ciudad                        14560 non-null  object
 10  tamaño_m2                     14560 non-null  int64 
 11  stock_actual                  14560 non-null  int64 
dtypes: int64(6), object(6)
memory usage: 1.3+ MB


In [11]:
df["fecha"] = pd.to_datetime(df["fecha"])

In [ ]:
df = df.sort_values(["id_tienda", "id_producto", "fecha"]).reset_index(drop=True)

In [13]:
df.head()

,fecha,id_tienda,id_producto,unidades_vendidas,nombre,categoria,costo_unitario,precio_venta,costo_almacenamiento_semanal,ciudad,tamaño_m2,stock_actual
0,2024-01-01,STORE_01,PROD_001,5,Tinto,Bebidas,800,2500,10,Bogotá,28,1
1,2024-01-02,STORE_01,PROD_001,4,Tinto,Bebidas,800,2500,10,Bogotá,28,1
2,2024-01-03,STORE_01,PROD_001,3,Tinto,Bebidas,800,2500,10,Bogotá,28,1
3,2024-01-04,STORE_01,PROD_001,4,Tinto,Bebidas,800,2500,10,Bogotá,28,1
4,2024-01-05,STORE_01,PROD_001,10,Tinto,Bebidas,800,2500,10,Bogotá,28,1


### Creación de nuevas variables
1. Variables temporales

In [15]:
df = create_time_features(df)

2. Variables historicas

In [23]:
df = create_lag_features(df)

3. Variables de ventana movil

In [24]:
df = create_rolling_features(df)

In [ ]:
df[["lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_std_7"]].isna().sum()

lag_1              160
lag_7             1120
lag_14            2240
rolling_mean_7    1120
rolling_std_7     1120
dtype: int64

In [29]:
df = clean_training_data(df)

### Tratamiento de valores nulos

La creación de variables históricas (lags y estadísticas móviles) genera valores nulos al inicio de cada serie temporal, ya que no existe suficiente historial para calcular dichas variables.

Dado que estos registros no contienen la información necesaria para el entrenamiento del modelo, se eliminan del conjunto de entrenamiento.

In [24]:
print(df.shape)
df.isna().sum().sum()

(12320, 23)


np.int64(0)

In [30]:
save_processed_data(df)

### Resumen

Se integraron las fuentes de datos de ventas, catálogo, tiendas e inventario.

Se generaron variables temporales derivadas de la fecha.

Se construyeron variables históricas (lag_1, lag_7, lag_14) y estadísticas móviles (rolling_mean_7, rolling_std_7) por combinación tienda–producto.

Se eliminaron las observaciones sin historial suficiente para evitar introducir sesgos en el entrenamiento.

El dataset final contiene 12.320 observaciones y 23 features listas para el modelado.